In [1]:
try:
    import google.colab

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q opendatasets
    !pip install -q mlflow
    !pip install -q torch
    !pip install -q numpy
    !pip install -q transformers
    !pip install -q torchinfo
    !pip install -q polars
    !pip install -q torchinfo
    !pip install -q tqdm
    !pip install -q scikit-learn
    !pip install -q matplotlib
    !pip install -q seaborn

In [2]:
import opendatasets as od

od.download(
    "https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection"
)

Skipping, found downloaded files in "./news-headlines-dataset-for-sarcasm-detection" (use force=True to force download)


In [3]:
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import polars as pl
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.modeling_bert import BertModel

# Types
from transformers.models.bert.tokenization_bert_fast import BertTokenizerFast

mlflow.set_tracking_uri("http://192.168.100.203:5000")
mlflow.set_experiment("[P] bert-sarcasm-detection")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
dataPath = (
    "./news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset.json"
)
data = pl.read_ndjson(dataPath)

dataset = mlflow.data.from_pandas(
    data.to_pandas(), name="Sarcasm Headlines Dataset", targets="is_sarcastic"
)
df = data.drop(["article_link"]).drop_nulls().drop_nans()
df

headline,is_sarcastic
str,i64
"""former versace store clerk sue…",0
"""the 'roseanne' revival catches…",0
"""mom starting to fear son's web…",1
"""boehner just wants wife to lis…",1
"""j.k. rowling wishes snape happ…",0
…,…
"""american politics in moral fre…",0
"""america's best 20 hikes""",0
"""reparations and obama""",0


In [5]:
X_validation, X_holdout, y_validation, y_holdout = train_test_split(
    df["headline"],
    df["is_sarcastic"],
    test_size=0.2,
    random_state=42,
)

X_train, X_test, y_train, y_test = train_test_split(
    X_validation,
    y_validation,
    test_size=0.2,
    random_state=42,
)

print(
    f"Train: {len(X_train)}, Test: {len(X_test)}, Validation: {len(X_validation)}, Holdout: {len(X_holdout)}"
)
print(
    f"Train ratio: {len(X_train) / len(df):.2f}, Test ratio: {len(X_test) / len(df):.2f}, Validation ratio: {len(X_validation) / len(df):.2f}, Holdout ratio: {len(X_holdout) / len(df):.2f}"
)

Train: 17093, Test: 4274, Validation: 21367, Holdout: 5342
Train ratio: 0.64, Test ratio: 0.16, Validation ratio: 0.80, Holdout ratio: 0.20


In [6]:
bert_model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(bert_model_name)
bert_model = AutoModel.from_pretrained(bert_model_name)

In [7]:
class SarcasmDataset(Dataset):
    def __init__(self, X: pl.Series, y: pl.Series, bert_tokenizer: BertTokenizerFast):
        assert isinstance(X, pl.Series), "X must be a polars DataFrame"
        assert isinstance(y, pl.Series), "y must be a polars DataFrame"
        assert len(X) == len(y), "X and y must have the same length"
        assert "headline" == X.name, "X must contain a 'headline' column"
        assert "is_sarcastic" == y.name, "y must contain an 'is_sarcastic' column"
        assert isinstance(bert_tokenizer, BertTokenizerFast), (
            "tokenizer must be an instance of AutoTokenizer"
        )
        self.X = [
            bert_tokenizer(
                x,
                max_length=100,
                truncation=True,
                padding="max_length",
                return_tensors="pt",
            )
            for x in X
        ]

        self.y = y.to_torch().to(torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        return x, y


training_dataset = SarcasmDataset(X_train, y_train, tokenizer)
validation_dataset = SarcasmDataset(X_test, y_test, tokenizer)
holdout_dataset = SarcasmDataset(X_holdout, y_holdout, tokenizer)

In [8]:
params = {
    "BATCH_SIZE": 32,
    "EPOCHS": 10,
    "LEARNING_RATE": 1e-4,
    "DROPOUT_RATE": 0.25,
}

In [9]:
train_dataloader = DataLoader(
    training_dataset, batch_size=params["BATCH_SIZE"], shuffle=True
)
validation_dataloader = DataLoader(
    validation_dataset, batch_size=params["BATCH_SIZE"], shuffle=False
)
holdout_dataloader = DataLoader(
    holdout_dataset, batch_size=params["BATCH_SIZE"], shuffle=False
)

In [10]:
class SarcasmDetector(nn.Module):
    def __init__(self, bert_model: BertModel, dropout_rate: float):
        super(SarcasmDetector, self).__init__()
        assert isinstance(bert_model, BertModel), (
            "bert_model must be an instance of AutoModel"
        )

        self.bert = bert_model
        self.linear1 = nn.Linear(bert_model.config.hidden_size, 384)
        self.dropout = nn.Dropout(dropout_rate)
        self.linear2 = nn.Linear(384, 1)
        self.activation = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        pooled_output = self.bert(
            input_ids=input_ids, attention_mask=attention_mask, return_dict=False
        )[0][:, 0]

        h = self.linear1(pooled_output)
        h = self.dropout(h)
        h = self.linear2(h)
        h = self.activation(h)

        return h

In [11]:
for param in bert_model.parameters():
    param.requires_grad = False

model = SarcasmDetector(bert_model, params["DROPOUT_RATE"])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=params["LEARNING_RATE"])

with mlflow.start_run(log_system_metrics=True, description="configuring setup", silent=True) as run:
    mlflow.log_params(params)
    mlflow.log_input(dataset, context="training", tags={"source": "Kaggle"})
    mlflow.set_tag("purpose", "practice")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("task-type", "text classification")
    mlflow.set_tag("task", "sarcasm detection")

    model.to(device)
    bert_model.to(device)
    for epoch in tqdm(range(params["EPOCHS"]), desc="Training epochs", unit="epoch"):

        ## Train and validate the model
        model.train()
        total_correct_train = 0
        total_samples_train = 0
        total_loss_train = 0

        total_correct_validation = 0
        total_samples_validation = 0
        total_loss_validation = 0

        for inputs, labels in tqdm(
            train_dataloader,
            desc=f"Epoch {epoch + 1}/{params['EPOCHS']} - Training batches",
            unit="batch",
            leave=False,
        ):
            inputs = inputs.to(device)
            labels = labels.to(device)

            predictions = model(
                input_ids=inputs["input_ids"].squeeze(1),
                attention_mask=inputs["attention_mask"].squeeze(1),
            ).squeeze(1)

            batch_loss = criterion(predictions, labels)
            total_loss_train += batch_loss.item()

            acc = (predictions.round() == labels).sum().item()
            total_correct_train += acc
            total_samples_train += labels.size(0)

            batch_loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        with torch.no_grad():
            model.eval()
            for inputs, labels in tqdm(
                validation_dataloader,
                desc=f"Epoch {epoch + 1}/{params['EPOCHS']} - Validation batches",
                unit="batch",
                leave=False,
            ):
                inputs = inputs.to(device)
                labels = labels.to(device)

                predictions = model(
                    input_ids=inputs["input_ids"].squeeze(1),
                    attention_mask=inputs["attention_mask"].squeeze(1),
                ).squeeze(1)

                batch_loss = criterion(predictions, labels)
                total_loss_validation += batch_loss.item()

                acc = (predictions.round() == labels).sum().item()
                total_correct_validation += acc
                total_samples_validation += labels.size(0)

        mlflow.log_metrics(
            {
                "train_loss": total_loss_train / len(train_dataloader),
                "train_accuracy": total_correct_train / total_samples_train,
                "validation_loss": total_loss_validation / len(validation_dataloader),
                "validation_accuracy": total_correct_validation
                / total_samples_validation,
            },
            step=epoch,
        )

    ## Evaluate on holdout set

    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():
        model.eval()
        for inputs, labels in tqdm(
            holdout_dataloader,
            desc="Holdout batches",
            unit="batch",
            leave=False,
        ):
            inputs = inputs.to(device)
            labels = labels.to(device)

            predictions = model(
                input_ids=inputs["input_ids"].squeeze(1),
                attention_mask=inputs["attention_mask"].squeeze(1),
            ).squeeze(1)

            batch_loss = criterion(predictions, labels)
            test_loss += batch_loss.item()

            acc = (predictions.round() == labels).sum().item()
            test_correct += acc
            test_total += labels.size(0)

    mlflow.log_metrics(
        {
            "holdout_loss": test_loss / len(holdout_dataloader),
            "holdout_accuracy": test_correct / test_total,
        }
    )

    input_data = {
        "input_ids": training_dataset[0][0]["input_ids"],
        "attention_mask": training_dataset[0][0]["attention_mask"],
    }

    with open("model_summary.txt", "w") as f:
        f.write(
            str(
                summary(
                    model,
                    device=str(device),
                    input_data=input_data,
                    col_names=[
                        "input_size",
                        "output_size",
                        "num_params",
                        "params_percent",
                        "mult_adds",
                        "kernel_size",
                    ],
                    verbose=1,
                )
            )
        )

    model.to("cpu")
    with open("model_architecture.txt", "w") as f:
        f.write(str(model))

    mlflow.pytorch.log_model(
        model,
        name="Sarcasm Detector",
        input_example={
            "input_ids": training_dataset[0][0]["input_ids"].squeeze(0).numpy(),
            "attention_mask": training_dataset[0][0]["attention_mask"]
            .squeeze(0)
            .numpy(),
        },
        extra_files=["model_summary.txt", "model_architecture.txt"],
    )

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
2025/07/05 08:03:20 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/07/05 08:03:20 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
/home/iragca/Documents/github/capstone-project-2/.venv/lib/python3.12/site-packages/mlflow/type

Training epochs:   0%|          | 0/10 [00:00<?, ?epoch/s]

Epoch 1/10 - Training batches:   0%|          | 0/535 [00:00<?, ?batch/s]

2025/07/05 08:03:31 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:03:41 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:03:51 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:04:01 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:04:11 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:04:21 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 08:04:31 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not S

Epoch 1/10 - Validation batches:   0%|          | 0/134 [00:00<?, ?batch/s]

...

2025/07/05 09:08:09 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:08:19 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:08:29 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:08:39 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:08:50 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:09:00 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:09:10 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not S

Layer (type:depth-idx)                                       Input Shape               Output Shape              Param #                   Param %                   Mult-Adds                 Kernel Shape
SarcasmDetector                                              --                        [1, 1]                    --                             --                   --                        --
├─BertModel: 1-1                                             --                        [1, 100, 768]             --                             --                   --                        --
│    └─BertEmbeddings: 2-1                                   --                        [1, 100, 768]             --                             --                   --                        --
│    │    └─Embedding: 3-1                                   [1, 100]                  [1, 100, 768]             (23,440,896)               21.35%                   23,440,896                --
│    │    └─Embeddin

2025/07/05 09:09:35 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: TypeError('The PyTorch flavor does not support List or Dict input types. Please use a pandas.DataFrame or a numpy.ndarray'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.


2025/07/05 09:09:40 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/07/05 09:09:40 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.
2025/07/05 09:09:40 WARNING mlflow.models.model: Failed to validate serving input example {
  "inputs": {
    "input_ids": [
      101,
      7658,
      6968,
      2912,
      2232,
      14529,
      2108,
      13366,
      10732,
      2094,
      3041,
      2296,
      2095,
      102,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,
      0,


🏃 View run serious-snail-225 at: http://192.168.100.203:5000/#/experiments/4/runs/fa3b989ed6b64cccb65937f9af17b7ba
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/4
